# Fine-tune MobileFaceNet on African Faces (FAGE_v2)

This notebook fine-tunes a pre-trained MobileFaceNet model on the **FAGE_v2 African dataset** to improve face verification accuracy for Nigerian field workers.

**Requirements**: T4 GPU runtime (Runtime > Change runtime type > T4 GPU)

**Output**: `mobilefacenet_african_v2.tflite` — drop-in replacement for the Flutter app

| Spec | Value |
|------|-------|
| Input | `[1, 112, 112, 3]` Float32, normalized `[-1, 1]` |
| Output | `[1, 192]` Float32, L2-normalized |
| Target | 90%+ accuracy on African face pairs |

## Cell 1 — Setup & Dependencies

In [ ]:
# Install dependencies
!pip install -q onnx2torch onnxruntime onnx-tf tensorflow \
    torch torchvision scikit-learn matplotlib tqdm onnx

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Mount Google Drive (stores the ONNX model + checkpoints)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/mobilefacenet_african'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f'Google Drive mounted. Save dir: {SAVE_DIR}')
except ImportError:
    SAVE_DIR = './checkpoints'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f'Not in Colab. Save dir: {SAVE_DIR}')

## Cell 2 — Download Pre-trained MobileFaceNet

In [ ]:
import onnx
import onnx2torch
import onnxruntime as ort
import shutil

# ============================================================
# Load MobileFaceNet ONNX model from Google Drive
#
# SETUP: Before running this cell, upload w600k_mbf.onnx to your
# Google Drive at: My Drive/mobilefacenet_african/w600k_mbf.onnx
#
# Download it once from:
#   https://huggingface.co/WePrompt/buffalo_sc/resolve/main/w600k_mbf.onnx
#
# Model spec: Input [1,3,112,112] NCHW -> Output [1,512] embedding
# We add a trainable projection 512->192 to match the Flutter app.
# ============================================================
ONNX_PATH = '/content/w600k_mbf.onnx'
DRIVE_ONNX_PATH = '/content/drive/MyDrive/mobilefacenet_african/w600k_mbf.onnx'

if not os.path.exists(ONNX_PATH):
    if os.path.exists(DRIVE_ONNX_PATH):
        shutil.copy(DRIVE_ONNX_PATH, ONNX_PATH)
        print(f'Copied from Google Drive: {DRIVE_ONNX_PATH}')
    else:
        raise FileNotFoundError(
            f'ONNX model not found at {DRIVE_ONNX_PATH}\n\n'
            'Please upload w600k_mbf.onnx to your Google Drive:\n'
            '  1. Download from: https://huggingface.co/WePrompt/buffalo_sc/resolve/main/w600k_mbf.onnx\n'
            '  2. Upload to: My Drive/mobilefacenet_african/w600k_mbf.onnx\n'
            '  3. Re-run this cell'
        )

print(f'ONNX model: {os.path.getsize(ONNX_PATH) / 1024 / 1024:.1f} MB')

# ============================================================
# Inspect the ONNX model
# ============================================================
onnx_model = onnx.load(ONNX_PATH)
input_info = onnx_model.graph.input[0]
output_info = onnx_model.graph.output[0]
input_shape = [d.dim_value for d in input_info.type.tensor_type.shape.dim]
output_shape = [d.dim_value for d in output_info.type.tensor_type.shape.dim]
print(f'Input:  name={input_info.name}, shape={input_shape}')
print(f'Output: name={output_info.name}, shape={output_shape}')

# w600k_mbf: NCHW [B,3,112,112] -> [1,512]
ONNX_EMBEDDING_DIM = output_shape[-1]
TARGET_EMBEDDING_DIM = 192  # Must match Flutter app
print(f'\nONNX embedding dim: {ONNX_EMBEDDING_DIM}')
print(f'Flutter target dim: {TARGET_EMBEDDING_DIM}')

# ============================================================
# Verify with ONNX Runtime
# ============================================================
sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
dummy_np = np.random.randn(1, 3, 112, 112).astype(np.float32)
onnx_output = sess.run(None, {input_info.name: dummy_np})[0]
print(f'ONNX Runtime: input {dummy_np.shape} -> output {onnx_output.shape}')

# ============================================================
# Convert ONNX to PyTorch
# ============================================================
print('\nConverting ONNX -> PyTorch...')
pytorch_model_raw = onnx2torch.convert(ONNX_PATH)
pytorch_model_raw = pytorch_model_raw.to(DEVICE)
pytorch_model_raw.eval()


class BackboneWrapper(nn.Module):
    """Wraps onnx2torch model to:
    1. Handle list/tuple outputs from onnx2torch
    2. Project from 512 -> 192 to match Flutter app
    """
    def __init__(self, raw_model, onnx_dim, target_dim):
        super().__init__()
        self.raw_model = raw_model
        self.onnx_dim = onnx_dim
        self.needs_projection = (onnx_dim != target_dim)
        if self.needs_projection:
            self.projection = nn.Linear(onnx_dim, target_dim, bias=False)
            nn.init.xavier_uniform_(self.projection.weight)
            print(f'  Added projection: {onnx_dim} -> {target_dim}')
        else:
            print(f'  No projection needed (dim={onnx_dim})')

    def forward(self, x):
        out = self.raw_model(x)
        # onnx2torch may return list/tuple of tensors
        if isinstance(out, (list, tuple)):
            for t in out:
                if isinstance(t, torch.Tensor) and t.dim() == 2 and t.shape[1] == self.onnx_dim:
                    out = t
                    break
            else:
                out = out[-1] if isinstance(out[-1], torch.Tensor) else out[0]
        if self.needs_projection:
            out = self.projection(out)
        return out


pytorch_model = BackboneWrapper(
    pytorch_model_raw,
    onnx_dim=ONNX_EMBEDDING_DIM,
    target_dim=TARGET_EMBEDDING_DIM,
).to(DEVICE)
pytorch_model.eval()

# ============================================================
# Verify: NCHW input -> 192-dim output
# ============================================================
with torch.no_grad():
    dummy_input = torch.randn(1, 3, 112, 112).to(DEVICE)
    dummy_output = pytorch_model(dummy_input)
    print(f'\nPyTorch wrapper: input (1,3,112,112) -> output {dummy_output.shape}')
    assert dummy_output.shape[1] == TARGET_EMBEDDING_DIM, \
        f'Expected dim {TARGET_EMBEDDING_DIM}, got {dummy_output.shape[1]}'
    print(f'Model verified: NCHW input -> {TARGET_EMBEDDING_DIM}-dim embedding')

# Cross-check raw model vs ONNX Runtime (pre-projection)
with torch.no_grad():
    raw_out = pytorch_model_raw(dummy_input)
    if isinstance(raw_out, (list, tuple)):
        for t in raw_out:
            if isinstance(t, torch.Tensor) and t.dim() == 2:
                raw_out = t
                break
    raw_out_np = raw_out.cpu().numpy()

cos_sim = np.dot(onnx_output.flatten(), raw_out_np.flatten()) / (
    np.linalg.norm(onnx_output) * np.linalg.norm(raw_out_np) + 1e-8)
print(f'ONNX Runtime vs PyTorch (raw) cosine sim: {cos_sim:.6f} (should be > 0.99)')

## Cell 3 — Load & Prepare FAGE African Face Dataset

In [ ]:
import zipfile
import shutil
import subprocess
from collections import defaultdict
from pathlib import Path

# ============================================================
# Load FAGE_v2 Dataset (African Faces for Age-Invariant Recognition)
#
# 500 African identities x 10 images each = 5,000 images
# Countries: Nigeria, Ghana, Kenya, Ethiopia, South Africa,
#            Egypt, Algeria, DR Congo, Angola, Namibia
# License: CC BY 4.0 (free for research)
# Source: https://www.kaggle.com/datasets/ajewoleolaitan/fage-dataset
#
# SETUP: Download from Kaggle and upload to Google Drive:
#   1. Go to https://www.kaggle.com/datasets/ajewoleolaitan/fage-dataset
#   2. Click "Download" (~53 MB zip)
#   3. Upload to: My Drive/mobilefacenet_african/fage_v2.zip
#      OR use Kaggle API: kaggle datasets download -d ajewoleolaitan/fage-dataset
# ============================================================
FAGE_DIR = '/content/fage_v2'
os.makedirs(FAGE_DIR, exist_ok=True)

DRIVE_FAGE_ZIP = '/content/drive/MyDrive/mobilefacenet_african/fage_v2.zip'
# Also check common Kaggle download name
DRIVE_FAGE_ZIP_ALT = '/content/drive/MyDrive/mobilefacenet_african/archive.zip'

if not any(Path(FAGE_DIR).rglob('*.jpg')) and not any(Path(FAGE_DIR).rglob('*.png')):
    extracted = False

    # Try Kaggle API first (works if kaggle.json is configured)
    if not extracted:
        try:
            print('Trying Kaggle API...')
            subprocess.run(
                ['kaggle', 'datasets', 'download', '-d', 'ajewoleolaitan/fage-dataset',
                 '-p', '/content/', '--unzip'],
                check=True, capture_output=True, timeout=120
            )
            # Move extracted content to FAGE_DIR
            for candidate in ['/content/fage-dataset', '/content/FAGE_v2', '/content/fage_v2']:
                if os.path.exists(candidate):
                    shutil.copytree(candidate, FAGE_DIR, dirs_exist_ok=True)
                    extracted = True
                    print(f'Downloaded via Kaggle API')
                    break
        except Exception as e:
            print(f'  Kaggle API not available: {e}')

    # Try Google Drive zip
    if not extracted:
        for zip_path in [DRIVE_FAGE_ZIP, DRIVE_FAGE_ZIP_ALT]:
            if os.path.exists(zip_path):
                print(f'Extracting from: {zip_path}')
                with zipfile.ZipFile(zip_path, 'r') as zf:
                    zf.extractall(FAGE_DIR)
                extracted = True
                print('Extraction complete.')
                break

    # Try Colab upload
    if not extracted:
        print('FAGE dataset not found.')
        print(f'  Checked: {DRIVE_FAGE_ZIP}')
        print(f'  Checked: {DRIVE_FAGE_ZIP_ALT}')
        print()
        print('Please download from Kaggle and upload:')
        print('  1. Go to https://www.kaggle.com/datasets/ajewoleolaitan/fage-dataset')
        print('  2. Click Download (~53 MB)')
        print('  3. Upload zip to: My Drive/mobilefacenet_african/fage_v2.zip')
        print('  4. Re-run this cell')
        print()
        try:
            from google.colab import files
            print('Or upload directly here:')
            uploaded = files.upload()
            for fname in uploaded:
                if fname.endswith('.zip'):
                    with zipfile.ZipFile(fname, 'r') as zf:
                        zf.extractall(FAGE_DIR)
                    extracted = True
                    print(f'Extracted {fname}')
                    break
        except Exception:
            raise FileNotFoundError(
                'FAGE dataset not found. Download from:\n'
                '  https://www.kaggle.com/datasets/ajewoleolaitan/fage-dataset'
            )

# ============================================================
# Discover dataset structure
# ============================================================
print(f'\nDiscovering dataset structure in {FAGE_DIR}...')

# Find all image files
all_images = []
for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
    all_images.extend(Path(FAGE_DIR).rglob(ext))
all_images = sorted(all_images)
print(f'Total image files found: {len(all_images)}')

# FAGE structure: typically organized by country/person or person folders
# Detect the structure by examining paths
# Group images by their parent directory (identity)
identity_map = defaultdict(list)
for img_path in all_images:
    # Use parent folder name as identity
    identity = img_path.parent.name
    # Skip if parent is the root extraction dir
    if str(img_path.parent) == FAGE_DIR:
        # Images might be named with identity prefix: "personName_01.jpg"
        stem = img_path.stem
        # Try to extract identity from filename (everything before last underscore/number)
        parts = stem.rsplit('_', 1)
        if len(parts) == 2:
            identity = parts[0]
        else:
            identity = stem
    identity_map[identity].append(str(img_path))

# Filter: only keep identities with 2+ images (needed for training)
identity_map = {k: v for k, v in identity_map.items() if len(v) >= 2}

print(f'Identities with 2+ images: {len(identity_map)}')
if identity_map:
    sample_id = list(identity_map.keys())[0]
    print(f'Sample identity: "{sample_id}" -> {len(identity_map[sample_id])} images')
    imgs_per_id = [len(v) for v in identity_map.values()]
    print(f'Images per identity: min={min(imgs_per_id)}, max={max(imgs_per_id)}, avg={sum(imgs_per_id)/len(imgs_per_id):.1f}')

IMG_DIR = FAGE_DIR  # Root for the dataset

# ============================================================
# Create verification pairs for evaluation
# (since FAGE doesn't come with pre-defined pairs like LFW/RFW)
# ============================================================
import itertools

def create_verification_pairs(identity_map, n_same=500, n_diff=500, seed=42):
    """Generate same-person and different-person pairs for evaluation."""
    rng = random.Random(seed)
    identities = list(identity_map.keys())

    # Same-person pairs: pick 2 random images from the same identity
    same_pairs = []
    ids_with_multiple = [k for k, v in identity_map.items() if len(v) >= 2]
    for _ in range(n_same):
        identity = rng.choice(ids_with_multiple)
        img1, img2 = rng.sample(identity_map[identity], 2)
        same_pairs.append((img1, img2, 1))

    # Different-person pairs: pick 1 image from 2 different identities
    diff_pairs = []
    for _ in range(n_diff):
        id1, id2 = rng.sample(identities, 2)
        img1 = rng.choice(identity_map[id1])
        img2 = rng.choice(identity_map[id2])
        diff_pairs.append((img1, img2, 0))

    return same_pairs + diff_pairs

verification_pairs = create_verification_pairs(identity_map, n_same=500, n_diff=500)
same_pairs = [p for p in verification_pairs if p[2] == 1]
diff_pairs = [p for p in verification_pairs if p[2] == 0]
print(f'\nVerification pairs: {len(verification_pairs)} total')
print(f'  Same person: {len(same_pairs)}')
print(f'  Different person: {len(diff_pairs)}')

# ============================================================
# Create training Dataset with augmentations
# ============================================================
class AfricanFaceDataset(Dataset):
    """Dataset for fine-tuning on African faces."""

    def __init__(self, identity_map, transform=None):
        self.transform = transform
        self.samples = []
        self.class_to_idx = {}

        for idx, (identity, img_paths) in enumerate(sorted(identity_map.items())):
            self.class_to_idx[identity] = idx
            for img_path in img_paths:
                self.samples.append((img_path, idx))

        self.num_classes = len(identity_map)
        print(f'Dataset: {len(self.samples)} images, {self.num_classes} classes')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        img_path, label = self.samples[index]
        img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)
        else:
            img = transforms.ToTensor()(img)

        return img, label


# Training augmentations
train_transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.1,
        hue=0.05,
    ),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# Evaluation transform (no augmentation)
eval_transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# Create dataset and dataloaders
train_dataset = AfricanFaceDataset(identity_map, transform=train_transform)

# Split into train/val (90/10)
from torch.utils.data import random_split
n_val = max(1, int(0.1 * len(train_dataset)))
n_train = len(train_dataset) - n_val
train_subset, val_subset = random_split(
    train_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(
    train_subset, batch_size=64, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_subset, batch_size=64, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f'\nTrain: {len(train_subset)} images')
print(f'Val: {len(val_subset)} images')
print(f'Batches per epoch: {len(train_loader)}')
print(f'Number of classes: {train_dataset.num_classes}')

## Cell 4 — Define ArcFace Loss + Training Loop

In [ ]:
import math
import torch.nn.functional as F


class ArcFaceLoss(nn.Module):
    """ArcFace (Additive Angular Margin) loss for face recognition.

    Adds an angular margin penalty to the target logit to enforce
    intra-class compactness and inter-class separability.

    Args:
        in_features: Embedding dimension (192 for MobileFaceNet)
        out_features: Number of classes (identities)
        s: Feature scale (default: 64.0)
        m: Angular margin in radians (default: 0.5)
    """

    def __init__(self, in_features, out_features, s=64.0, m=0.5):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)  # threshold
        self.mm = math.sin(math.pi - m) * m  # for numerical stability

        # Class weight matrix (learnable)
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings, labels):
        # Normalize embeddings and weights
        embeddings = F.normalize(embeddings, p=2, dim=1)
        weights = F.normalize(self.weight, p=2, dim=1)

        # Cosine similarity
        cosine = F.linear(embeddings, weights)
        sine = torch.sqrt(1.0 - torch.clamp(cosine * cosine, 0, 1))

        # cos(theta + m) = cos(theta)*cos(m) - sin(theta)*sin(m)
        phi = cosine * self.cos_m - sine * self.sin_m

        # Numerical stability: when cos(theta) < cos(pi - m), use fallback
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        # One-hot encode labels
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)

        # Apply margin only to the target class
        logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        logits *= self.s

        loss = F.cross_entropy(logits, labels)
        return loss


class FinetuneModel(nn.Module):
    """Wrapper that combines backbone + ArcFace head for training."""

    def __init__(self, backbone, embedding_dim=192, num_classes=100):
        super().__init__()
        self.backbone = backbone
        self.arcface = ArcFaceLoss(
            in_features=embedding_dim,
            out_features=num_classes,
            s=64.0,
            m=0.5,
        )

    def forward(self, images, labels=None):
        embeddings = self.backbone(images)
        if labels is not None:
            loss = self.arcface(embeddings, labels)
            return embeddings, loss
        return embeddings


# ============================================================
# Freeze strategy:
#   - Freeze early layers of the raw_model (CNN backbone)
#   - Keep projection layer (if present) fully trainable
#   - Keep ArcFace head fully trainable
# ============================================================
def freeze_early_layers(model, freeze_ratio=0.5):
    """Freeze the first `freeze_ratio` of the raw CNN backbone parameters."""
    # Only freeze inside the raw_model (the onnx2torch CNN), not projection
    raw_params = list(model.backbone.raw_model.parameters())
    n_freeze = int(len(raw_params) * freeze_ratio)

    for i, param in enumerate(raw_params):
        param.requires_grad = (i >= n_freeze)

    # Ensure projection layer is always trainable
    if model.backbone.needs_projection:
        for param in model.backbone.projection.parameters():
            param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')
    print(f'Frozen: first {n_freeze} / {len(raw_params)} raw backbone param groups')
    if model.backbone.needs_projection:
        proj_params = sum(p.numel() for p in model.backbone.projection.parameters())
        print(f'Projection layer ({proj_params:,} params): always trainable')


# Build the fine-tuning model
model = FinetuneModel(
    backbone=pytorch_model,
    embedding_dim=TARGET_EMBEDDING_DIM,
    num_classes=train_dataset.num_classes,
).to(DEVICE)

freeze_early_layers(model, freeze_ratio=0.5)

# Optimizer: SGD with momentum (standard for face recognition fine-tuning)
optimizer = optim.SGD(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    momentum=0.9,
    weight_decay=5e-4,
)

# Learning rate schedule: cosine annealing
NUM_EPOCHS = 20
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print(f'\nTraining config:')
print(f'  Epochs: {NUM_EPOCHS}')
print(f'  Batch size: 64')
print(f'  Optimizer: SGD (lr=1e-3, momentum=0.9, wd=5e-4)')
print(f'  Scheduler: CosineAnnealingLR')
print(f'  Loss: ArcFace (s=64, m=0.5)')
print(f'  Backbone embedding dim: {ONNX_EMBEDDING_DIM}')
print(f'  Output embedding dim: {TARGET_EMBEDDING_DIM}')

## Cell 5 — Fine-tuning Execution

In [ ]:
# ============================================================
# Training loop
# ============================================================
train_losses = []
val_losses = []
learning_rates = []
best_val_loss = float('inf')
best_epoch = 0

print('Starting fine-tuning...\n')

for epoch in range(NUM_EPOCHS):
    # --- Training ---
    model.train()
    epoch_loss = 0.0
    n_batches = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for images, labels in pbar:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        _, loss = model(images, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.6f}'})

    avg_train_loss = epoch_loss / max(n_batches, 1)
    train_losses.append(avg_train_loss)
    learning_rates.append(scheduler.get_last_lr()[0])

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_batches = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            _, loss = model(images, labels)
            val_loss += loss.item()
            val_batches += 1

    avg_val_loss = val_loss / max(val_batches, 1)
    val_losses.append(avg_val_loss)

    print(f'  Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, '
          f'val_loss={avg_val_loss:.4f}, lr={scheduler.get_last_lr()[0]:.6f}')

    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch + 1
        torch.save({
            'epoch': epoch + 1,
            'backbone_state_dict': model.backbone.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': avg_val_loss,
            'train_loss': avg_train_loss,
        }, os.path.join(SAVE_DIR, 'best_mobilefacenet_african.pth'))
        print(f'  -> Saved best model (epoch {epoch+1}, val_loss={avg_val_loss:.4f})')

    scheduler.step()

print(f'\nTraining complete. Best model: epoch {best_epoch} (val_loss={best_val_loss:.4f})')

# ============================================================
# Plot training curves
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, NUM_EPOCHS+1), train_losses, 'b-', label='Train Loss')
ax1.plot(range(1, NUM_EPOCHS+1), val_losses, 'r-', label='Val Loss')
ax1.axvline(best_epoch, color='g', linestyle='--', label=f'Best (epoch {best_epoch})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(range(1, NUM_EPOCHS+1), learning_rates, 'g-')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Learning Rate')
ax2.set_title('Learning Rate Schedule')
ax2.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=150)
plt.show()

## Cell 6 — Evaluation on FAGE African Verification Pairs

In [ ]:
# ============================================================
# Load best checkpoint and evaluate on verification pairs
# ============================================================
best_ckpt = torch.load(
    os.path.join(SAVE_DIR, 'best_mobilefacenet_african.pth'),
    map_location=DEVICE,
)
model.backbone.load_state_dict(best_ckpt['backbone_state_dict'])
model.eval()
print(f'Loaded best model from epoch {best_ckpt["epoch"]} (val_loss={best_ckpt["val_loss"]:.4f})')


def extract_embedding(model, img_path, transform):
    """Extract a 192-dim embedding from an image."""
    img = Image.open(img_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        embedding = model.backbone(img_tensor)
        embedding = F.normalize(embedding, p=2, dim=1)
    return embedding.cpu().numpy().flatten()


# Compute embeddings and similarities for all verification pairs
print(f'\nEvaluating on {len(verification_pairs)} verification pairs...')
similarities = []
labels = []

for path1, path2, label in tqdm(verification_pairs, desc='Extracting embeddings'):
    try:
        emb1 = extract_embedding(model, path1, eval_transform)
        emb2 = extract_embedding(model, path2, eval_transform)
        sim = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
        similarities.append(sim)
        labels.append(label)
    except Exception as e:
        continue  # Skip corrupted images

similarities = np.array(similarities)
labels = np.array(labels)
print(f'Successfully evaluated {len(similarities)} pairs')

# ============================================================
# ROC Analysis
# ============================================================
fpr, tpr, thresholds = roc_curve(labels, similarities)
roc_auc = auc(fpr, tpr)

# Find optimal threshold (maximizes TPR - FPR, i.e., Youden's J statistic)
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]

# Find threshold at FAR < 1%
far_1_idx = np.where(fpr <= 0.01)[0]
if len(far_1_idx) > 0:
    threshold_at_far1 = thresholds[far_1_idx[-1]]
    tpr_at_far1 = tpr[far_1_idx[-1]]
else:
    threshold_at_far1 = optimal_threshold
    tpr_at_far1 = 0.0

# Compute metrics at optimal threshold
predictions = (similarities >= optimal_threshold).astype(int)
accuracy = np.mean(predictions == labels)
tp = np.sum((predictions == 1) & (labels == 1))
fp = np.sum((predictions == 1) & (labels == 0))
tn = np.sum((predictions == 0) & (labels == 0))
fn = np.sum((predictions == 0) & (labels == 1))

far = fp / max(fp + tn, 1)  # False Accept Rate
frr = fn / max(fn + tp, 1)  # False Reject Rate

print(f'\n{"="*60}')
print(f' EVALUATION RESULTS — FAGE African Verification Pairs')
print(f'{"="*60}')
print(f' AUC:                  {roc_auc:.4f}')
print(f' Optimal threshold:    {optimal_threshold:.4f}')
print(f' Accuracy:             {accuracy*100:.2f}%')
print(f' FAR:                  {far*100:.2f}%')
print(f' FRR:                  {frr*100:.2f}%')
print(f'{"="*60}')
print(f' At FAR < 1%:')
print(f'   Threshold:          {threshold_at_far1:.4f}')
print(f'   TPR (recall):       {tpr_at_far1*100:.2f}%')
print(f'{"="*60}')

# Recommended threshold for the Flutter app
# Use the FAR<1% threshold for security-sensitive field deployment
RECOMMENDED_THRESHOLD = round(float(threshold_at_far1), 2)
print(f'\n>>> RECOMMENDED THRESHOLD for Flutter app: {RECOMMENDED_THRESHOLD}')
print(f'>>> Update defaultThreshold in distance_metrics.dart to: {RECOMMENDED_THRESHOLD}')

# ============================================================
# Plot ROC curve and similarity distributions
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ROC curve
ax1.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax1.scatter(fpr[optimal_idx], tpr[optimal_idx], c='red', s=100, zorder=5,
            label=f'Optimal (t={optimal_threshold:.3f})')
ax1.set_xlabel('False Positive Rate (FAR)')
ax1.set_ylabel('True Positive Rate (1-FRR)')
ax1.set_title('ROC Curve — FAGE African Face Verification')
ax1.legend(loc='lower right')
ax1.grid(True)

# Similarity distributions
same_sims = similarities[labels == 1]
diff_sims = similarities[labels == 0]
ax2.hist(same_sims, bins=50, alpha=0.7, color='green', label='Same person', density=True)
ax2.hist(diff_sims, bins=50, alpha=0.7, color='red', label='Different person', density=True)
ax2.axvline(optimal_threshold, color='blue', linestyle='--', linewidth=2,
            label=f'Threshold = {optimal_threshold:.3f}')
ax2.set_xlabel('Cosine Similarity')
ax2.set_ylabel('Density')
ax2.set_title('Similarity Distribution — FAGE African Faces')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'evaluation_results.png'), dpi=150)
plt.show()

## Cell 7 — Export to TFLite

**Important:** If you get a numpy/tensorflow error, click **Runtime > Restart session** first, then run this cell. It is fully self-contained and will reload the model from the saved checkpoint.

In [ ]:
# ============================================================
# Self-contained TFLite export cell
# Run this after Runtime > Restart session if you hit numpy errors
# ============================================================
!pip install -q onnx2torch onnxruntime onnx onnxscript onnx2tf onnx_graphsurgeon sng4onnx

import os
import shutil
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import onnxruntime as ort
import tensorflow as tf

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/mobilefacenet_african'
except ImportError:
    SAVE_DIR = './checkpoints'

TARGET_EMBEDDING_DIM = 192

# ============================================================
# Step 0: Recreate model from ONNX + load fine-tuned weights
# ============================================================
print('\nStep 0: Rebuilding model from checkpoint...')

# Load ONNX
ONNX_PATH = '/content/w600k_mbf.onnx'
DRIVE_ONNX_PATH = os.path.join(SAVE_DIR, 'w600k_mbf.onnx')
if not os.path.exists(ONNX_PATH):
    if os.path.exists(DRIVE_ONNX_PATH):
        shutil.copy(DRIVE_ONNX_PATH, ONNX_PATH)
    else:
        raise FileNotFoundError(f'ONNX not found at {DRIVE_ONNX_PATH}')

onnx_model = onnx.load(ONNX_PATH)
ONNX_EMBEDDING_DIM = [d.dim_value for d in onnx_model.graph.output[0].type.tensor_type.shape.dim][-1]
print(f'  ONNX embedding dim: {ONNX_EMBEDDING_DIM}')

import onnx2torch
pytorch_model_raw = onnx2torch.convert(ONNX_PATH)
pytorch_model_raw = pytorch_model_raw.to(DEVICE).eval()


class BackboneWrapper(nn.Module):
    def __init__(self, raw_model, onnx_dim, target_dim):
        super().__init__()
        self.raw_model = raw_model
        self.onnx_dim = onnx_dim
        self.needs_projection = (onnx_dim != target_dim)
        if self.needs_projection:
            self.projection = nn.Linear(onnx_dim, target_dim, bias=False)

    def forward(self, x):
        out = self.raw_model(x)
        if isinstance(out, (list, tuple)):
            for t in out:
                if isinstance(t, torch.Tensor) and t.dim() == 2 and t.shape[1] == self.onnx_dim:
                    out = t
                    break
            else:
                out = out[-1] if isinstance(out[-1], torch.Tensor) else out[0]
        if self.needs_projection:
            out = self.projection(out)
        return out


backbone = BackboneWrapper(pytorch_model_raw, ONNX_EMBEDDING_DIM, TARGET_EMBEDDING_DIM).to(DEVICE)

# Load fine-tuned weights
ckpt_path = os.path.join(SAVE_DIR, 'best_mobilefacenet_african.pth')
ckpt = torch.load(ckpt_path, map_location=DEVICE)
backbone.load_state_dict(ckpt['backbone_state_dict'])
backbone.eval()
print(f'  Loaded checkpoint from epoch {ckpt["epoch"]} (val_loss={ckpt["val_loss"]:.4f})')

# Verify
with torch.no_grad():
    test_out = backbone(torch.randn(1, 3, 112, 112).to(DEVICE))
    print(f'  Output shape: {test_out.shape} (expected [1, {TARGET_EMBEDDING_DIM}])')
    assert test_out.shape[1] == TARGET_EMBEDDING_DIM

# ============================================================
# Step 1: Export PyTorch -> ONNX (legacy exporter keeps weights inline)
# ============================================================
print('\nStep 1: PyTorch -> ONNX')
dummy_input = torch.randn(1, 3, 112, 112).to(DEVICE)
ONNX_EXPORT_PATH = '/content/mobilefacenet_african_v2.onnx'

torch.onnx.export(
    backbone, dummy_input, ONNX_EXPORT_PATH,
    input_names=['input'], output_names=['embedding'],
    dynamic_axes={'input': {0: 'batch_size'}, 'embedding': {0: 'batch_size'}},
    opset_version=17, do_constant_folding=True,
    dynamo=False,  # Force legacy exporter — keeps weights inline
)

export_size = os.path.getsize(ONNX_EXPORT_PATH) / 1024 / 1024
print(f'  Exported: {export_size:.1f} MB')

# Verify ONNX matches PyTorch
with torch.no_grad():
    pt_out = backbone(dummy_input).cpu().numpy()
sess = ort.InferenceSession(ONNX_EXPORT_PATH, providers=['CPUExecutionProvider'])
onnx_out = sess.run(None, {'input': dummy_input.cpu().numpy()})[0]
cos_sim = np.dot(pt_out.flatten(), onnx_out.flatten()) / (
    np.linalg.norm(pt_out) * np.linalg.norm(onnx_out) + 1e-8)
print(f'  PyTorch vs ONNX cosine sim: {cos_sim:.6f}')
assert cos_sim > 0.99, f'ONNX export mismatch! cosine sim = {cos_sim}'

# ============================================================
# Step 2: ONNX -> TFLite via onnx2tf CLI
# ============================================================
print('\nStep 2: ONNX -> TFLite (via onnx2tf)')
TFLITE_OUTPUT_DIR = '/content/mobilefacenet_african_v2_tflite'
if os.path.exists(TFLITE_OUTPUT_DIR):
    shutil.rmtree(TFLITE_OUTPUT_DIR)

os.system(f'onnx2tf -i {ONNX_EXPORT_PATH} -o {TFLITE_OUTPUT_DIR} -oiqt -qt per-tensor --non_verbose')

# Find generated TFLite files
tflite_files = glob.glob(os.path.join(TFLITE_OUTPUT_DIR, '*.tflite'))
print(f'  Generated: {tflite_files}')

TFLITE_PATH = None
for f in sorted(tflite_files):
    if 'float32' in f:
        TFLITE_PATH = f
        break
if TFLITE_PATH is None and tflite_files:
    TFLITE_PATH = tflite_files[0]
if TFLITE_PATH is None:
    raise FileNotFoundError(f'No TFLite file in {TFLITE_OUTPUT_DIR}. Check onnx2tf output above.')

tflite_size_mb = os.path.getsize(TFLITE_PATH) / 1024 / 1024
print(f'  Using: {TFLITE_PATH} ({tflite_size_mb:.2f} MB)')

# ============================================================
# Step 3: Validate TFLite model
# ============================================================
print('\nStep 3: Validating TFLite...')
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
input_shape = input_details[0]['shape'].tolist()
output_shape = output_details[0]['shape'].tolist()

print(f'  Input:  {input_shape} ({input_details[0]["dtype"]})')
print(f'  Output: {output_shape} ({output_details[0]["dtype"]})')
print(f'  Layout: {"NHWC" if input_shape[-1] == 3 else "NCHW"}')

# Test inference
test_input = np.random.randn(*input_details[0]['shape']).astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_input)
interpreter.invoke()
tflite_output = interpreter.get_tensor(output_details[0]['index'])
print(f'  L2 norm: {np.linalg.norm(tflite_output):.4f}')

# Compare with PyTorch
if input_shape[-1] == 3:
    pt_input = np.transpose(test_input, (0, 3, 1, 2))
else:
    pt_input = test_input
with torch.no_grad():
    pt_out2 = backbone(torch.from_numpy(pt_input).to(DEVICE))
    pt_out2 = F.normalize(pt_out2, p=2, dim=1).cpu().numpy()
cos_sim2 = np.dot(tflite_output.flatten(), pt_out2.flatten()) / (
    np.linalg.norm(tflite_output) * np.linalg.norm(pt_out2) + 1e-8)
print(f'  TFLite vs PyTorch cosine sim: {cos_sim2:.6f}')

# ============================================================
# Step 4: Copy final model
# ============================================================
FINAL_TFLITE_PATH = '/content/mobilefacenet_african_v2.tflite'

if tflite_size_mb > 5.0:
    print(f'\n  Model > 5MB, applying float16 quantization...')
    saved_model_path = os.path.join(TFLITE_OUTPUT_DIR, 'saved_model')
    if not os.path.exists(saved_model_path):
        saved_model_path = TFLITE_OUTPUT_DIR
    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    tflite_q = converter.convert()
    with open(FINAL_TFLITE_PATH, 'wb') as f:
        f.write(tflite_q)
    tflite_size_mb = os.path.getsize(FINAL_TFLITE_PATH) / 1024 / 1024
    print(f'  Quantized: {tflite_size_mb:.2f} MB')
else:
    shutil.copy(TFLITE_PATH, FINAL_TFLITE_PATH)

print(f'\n{"="*60}')
print(f'  EXPORT COMPLETE')
print(f'{"="*60}')
print(f'  File: {FINAL_TFLITE_PATH}')
print(f'  Size: {os.path.getsize(FINAL_TFLITE_PATH)/1024/1024:.2f} MB')
print(f'  Input:  {input_shape}')
print(f'  Output: {output_shape}')
if input_shape == [1, 112, 112, 3] and output_shape[-1] == TARGET_EMBEDDING_DIM:
    print(f'  STATUS: PASSED — matches Flutter app spec')
else:
    print(f'  WARNING: Expected [1,112,112,3] -> [1,{TARGET_EMBEDDING_DIM}]')

# Save to Drive
drive_tflite = os.path.join(SAVE_DIR, 'mobilefacenet_african_v2.tflite')
shutil.copy(FINAL_TFLITE_PATH, drive_tflite)
print(f'\n  Saved to Drive: {drive_tflite}')

## Cell 8 — Download & Deploy

In [ ]:
import shutil

# ============================================================
# Copy to Google Drive for persistence
# ============================================================
drive_tflite = os.path.join(SAVE_DIR, 'mobilefacenet_african_v2.tflite')
shutil.copy(TFLITE_PATH, drive_tflite)
print(f'Model saved to Google Drive: {drive_tflite}')

# Also save as mobilefacenet.tflite (drop-in replacement)
drive_dropin = os.path.join(SAVE_DIR, 'mobilefacenet.tflite')
shutil.copy(TFLITE_PATH, drive_dropin)
print(f'Drop-in replacement: {drive_dropin}')

# ============================================================
# Download in Colab
# ============================================================
try:
    from google.colab import files
    print('\nDownloading TFLite model...')
    files.download(TFLITE_PATH)
except ImportError:
    print(f'\nNot in Colab. Model is at: {TFLITE_PATH}')

# ============================================================
# Deployment instructions
# ============================================================
print(f'''
{'='*60}
 DEPLOYMENT INSTRUCTIONS
{'='*60}

1. Copy the model file:
   cp mobilefacenet_african_v2.tflite \\
     packages/digit_face_verification/assets/models/mobilefacenet.tflite

2. Update distance_metrics.dart:
   static const double defaultThreshold = {RECOMMENDED_THRESHOLD};

3. Update face_embedding_model.dart:
   this.modelVersion = 'mobilefacenet_african_v2'

4. Update face_embedding_repository.dart:
   String modelVersion = 'mobilefacenet_african_v2'

5. IMPORTANT: Re-register all faces after model update.
   Embeddings from v1 are NOT compatible with v2.

{'='*60}
 MODEL PERFORMANCE SUMMARY
{'='*60}
 Accuracy:           {accuracy*100:.2f}%
 AUC:                {roc_auc:.4f}
 FAR:                {far*100:.2f}%
 FRR:                {frr*100:.2f}%
 Optimal threshold:  {optimal_threshold:.4f}
 Recommended (app):  {RECOMMENDED_THRESHOLD}
 Model size:         {tflite_size_mb:.2f} MB
 Model version:      mobilefacenet_african_v2
{'='*60}
''')